In [1]:
from pathlib import Path
from moviepy import VideoFileClip

def extract_audio(video_path, output_dir, sample_rate=16000, overwrite=True):
    """Extract mono WAV audio from a video file."""
    video_path = Path(video_path)
    output_dir = Path(output_dir)

    if not video_path.exists():
        raise FileNotFoundError(f"Video file not found: {video_path}")

    output_dir.mkdir(parents=True, exist_ok=True)
    audio_path = output_dir / f"{video_path.stem}.wav"

    if audio_path.exists() and not overwrite:
        return str(audio_path)

    with VideoFileClip(str(video_path)) as video:
        if video.audio is None:
            raise ValueError(f"No audio track found in: {video_path}")

        video.audio.write_audiofile(
            str(audio_path),
            fps=sample_rate,
            codec="pcm_s16le",
            ffmpeg_params=["-ac", "1"],
            logger=None,
        )

    return str(audio_path)

In [2]:
import whisper

_WHISPER_MODEL_CACHE = {}

def get_whisper_model(model_name="base"):
    """Load and cache Whisper models to avoid repeated heavy initialization."""
    if model_name not in _WHISPER_MODEL_CACHE:
        _WHISPER_MODEL_CACHE[model_name] = whisper.load_model(model_name)
    return _WHISPER_MODEL_CACHE[model_name]

def transcribe_audio(audio_path, model_name="base", language=None, task="transcribe"):
    """Transcribe audio and return transcript with metadata."""
    model = get_whisper_model(model_name)
    result = model.transcribe(
        audio_path,
        language=language,
        task=task,
        fp16=False,
    )

    transcript = result.get("text", "").strip()
    return {
        "transcript": transcript,
        "segments": result.get("segments", []),
        "language": result.get("language", language),
        "transcribe_result": result,
    }

In [3]:
import librosa
import numpy as np

def extract_audio_features(audio_path, sample_rate=16000, n_mfcc=13):
    """Extract robust time/frequency-domain audio features."""
    y, sr = librosa.load(audio_path, sr=sample_rate, mono=True)

    if y.size == 0:
        raise ValueError(f"Loaded empty audio: {audio_path}")

    duration = float(librosa.get_duration(y=y, sr=sr))
    rms = librosa.feature.rms(y=y)[0]
    zcr = librosa.feature.zero_crossing_rate(y=y)[0]
    centroid = librosa.feature.spectral_centroid(y=y, sr=sr)[0]
    bandwidth = librosa.feature.spectral_bandwidth(y=y, sr=sr)[0]
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)
    tempo, _ = librosa.beat.beat_track(y=y, sr=sr)
    tempo = float(np.atleast_1d(tempo)[0])

    features = {
        "audio_duration_seconds": duration,
        "sample_rate": int(sr),
        "voice_energy_mean": float(np.mean(rms)),
        "voice_energy_std": float(np.std(rms)),
        "zero_crossing_rate_mean": float(np.mean(zcr)),
        "spectral_centroid_mean": float(np.mean(centroid)),
        "spectral_bandwidth_mean": float(np.mean(bandwidth)),
        "tempo_bpm": tempo,
    }

    for idx in range(n_mfcc):
        features[f"mfcc_{idx + 1}_mean"] = float(np.mean(mfcc[idx]))
        features[f"mfcc_{idx + 1}_std"] = float(np.std(mfcc[idx]))

    return features

In [4]:
import json
import re
from pathlib import Path

def speech_rate(transcript, duration):
    """Calculate speech rate as words per minute with safe guards."""
    words = re.findall(r"[A-Za-z0-9']+", transcript or "")
    word_count = len(words)

    if duration is None or duration <= 0:
        return {
            "word_count": word_count,
            "words_per_minute": 0.0,
        }

    words_per_minute = float((word_count / duration) * 60.0)
    return {
        "word_count": word_count,
        "words_per_minute": words_per_minute,
    }

def run_audio_pipeline(video_path, output_dir, model_name="base", language=None):
    """End-to-end pipeline: video -> audio -> transcript -> features -> speech rate."""
    audio_path = extract_audio(video_path, output_dir)
    transcript_result = transcribe_audio(
        audio_path=audio_path,
        model_name=model_name,
        language=language,
    )
    audio_features = extract_audio_features(audio_path)
    rate = speech_rate(
        transcript_result["transcript"],
        audio_features["audio_duration_seconds"],
    )

    return {
        "video_path": str(video_path),
        "audio_path": audio_path,
        "transcript": transcript_result["transcript"],
        "language": transcript_result["language"],
        "word_count": rate["word_count"],
        "speech_rate_wpm": rate["words_per_minute"],
        "audio_features": audio_features,
        "segments": transcript_result["segments"],
    }

def save_pipeline_result(result, output_dir, file_name="audio_pipeline_result.json"):
    """Save pipeline output to JSON."""
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    output_path = output_dir / file_name

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(result, f, indent=2, ensure_ascii=False)

    return str(output_path)

In [9]:
from pathlib import Path
import os
import imageio_ffmpeg

video_path = r"E:\resume_project\ai-interview-analyzer\videoplayback.mp4"
output_dir = r"E:\resume_project\ai-interview-analyzer\data\processed\audio_pipeline"

# Ensure FFmpeg is available via a shim so Whisper can find "ffmpeg" by name
ffmpeg_path = imageio_ffmpeg.get_ffmpeg_exe()
ffmpeg_shim_dir = Path(output_dir)
ffmpeg_shim_dir.mkdir(parents=True, exist_ok=True)
ffmpeg_bat = ffmpeg_shim_dir / "ffmpeg.bat"
ffmpeg_bat.write_text(f'@"{ffmpeg_path}" %*\n')
ffmpeg_exe = ffmpeg_shim_dir / "ffmpeg.exe"
if not ffmpeg_exe.exists():
    import shutil
    shutil.copy2(ffmpeg_path, str(ffmpeg_exe))
os.environ["PATH"] = str(ffmpeg_shim_dir) + os.pathsep + os.environ["PATH"]

if not Path(video_path).exists():
    print("Video not found")
else:

    pipeline_result = run_audio_pipeline(
        video_path=str(Path(video_path).resolve()),
        output_dir=output_dir,
        model_name="base",
        language="en"
    )

    result_json_path = save_pipeline_result(pipeline_result, output_dir)

    print("Pipeline completed.")
    print(f"Audio file: {pipeline_result['audio_path']}")
    print(f"Transcript length: {len(pipeline_result['transcript'])} characters")
    print(f"Word count: {pipeline_result['word_count']}")
    print(f"Speech rate: {pipeline_result['speech_rate_wpm']:.2f} WPM")
    print(f"Saved result: {result_json_path}")

c:\Users\gk480\anaconda3\envs\deep_learning\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Pipeline completed.
Audio file: E:\resume_project\ai-interview-analyzer\data\processed\audio_pipeline\videoplayback.wav
Transcript length: 774 characters
Word count: 149
Speech rate: 192.51 WPM
Saved result: E:\resume_project\ai-interview-analyzer\data\processed\audio_pipeline\audio_pipeline_result.json
